In [1]:
import pandas as pd
import glob
import subprocess
import time
from SPARQLWrapper import SPARQLWrapper, TURTLE, JSON, CSV
import os

In [2]:
dico_top_pathway_ids = {
    "01": "Autophagy",
    "02": "CellCycle",
    "03": "CellCellCommunication",
    "04": "CellularResponseToStimuli",
    "05": "ChromatinOrganization",
    "06": "CircadianClock",
    "07": "DevelopmentalBiology",
    "08": "DigestionAndAbsorption",
    "09": "Disease",
    "10": "DNARepair",
    "11": "DNAReplication",
    "12": "DrugADME",
    "13": "ExtracellularMatrixOrganization",
    "14": "GeneExpression(Transcription)",
    "15": "Hemostasis",
    "16": "ImmuneSystem",
    "17": "Metabolism",
    "18": "MetabolismOfProteins",
    "19": "MetabolismOfRNA",
    "20": "MuscleContraction",
    "21": "NeuronalSystem",
    "22": "OrganelleBiogenesisAndMaintenance",
    "23": "ProgrammedCellDeath",
    "24": "ProteinLocalization",
    "25": "Reproduction",
    "26": "SensoryPerception",
    "27": "SignalTransduction",
    "28": "TransportOfSmallMolecules",
    "29": "VesicleMediatedTransport"
}

In [3]:
query = """
SELECT (COUNT(DISTINCT ?entityID) AS ?nbID)
WHERE {
    VALUES ?db { "UniProt" "UniProt Isoform" }

    ?pathway rdf:type bp3:Pathway .
    ?pathway (bp3:pathwayComponent | bp3:pathwayOrder/bp3:stepProcess)* ?interaction .
    ?interaction rdf:type/(rdfs:subClassOf*) bp3:Interaction .
    ?pathway bp3:xref ?pathwayXref .
    ?pathwayXref rdf:type bp3:UnificationXref ;
                    bp3:db "Reactome" ;
                    bp3:id ?pathwayID .

    VALUES ?relation { bp3:left bp3:right bp3:participant bp3:controller }
    ?interaction ?relation ?entity .
    
    ?entity (bp3:component | bp3:memberPhysicalEntity)* ?entityCompo .

    ?entityCompo rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
    ?entityCompo bp3:entityReference ?entityRef .
    ?entityRef bp3:xref ?entityRefXref .
    ?entityRefXref rdf:type bp3:UnificationXref ;
                    bp3:db ?db ;
                    bp3:id ?entityID .
}
"""

In [4]:
prefixes = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#>
"""
endpoint = "http://localhost:3030/reactome"
current_dir = os.getcwd()
BIOPAX_ONTOLOGY = os.path.join(current_dir, "../../Data/BioPAX/BioPAXOntology/biopax-level3.owl")

In [5]:
csv_path = '../../Results/PathwayComembership/02_WeightedComembershipCliques/'
csv_files = glob.glob(csv_path + '/*.csv')
results_df = pd.DataFrame(columns=["top_pathway", "nb_UniProt", "clique_size", "nb_scoresMICA", "nb_scoresNSP", "nb_scoresHybrid", "% NSP scores"])

counter = 1
i = 0
for file in sorted(csv_files):

    top_pathway_name = dico_top_pathway_ids[f"{counter:02d}"]
    print(top_pathway_name)
    results_df.at[i, "top_pathway"] = top_pathway_name

    # biopax = f"../../Data/BioPAX/ReactomeTopPathways/{counter:02d}_{top_pathway_name}.xml"
    # command = [
    #     '/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server',
    #     '--file', biopax,
    #     '--file', BIOPAX_ONTOLOGY,
    #     '/reactome'
    # ]
    # print("Fuseki command:", command)
    # process = subprocess.Popen(command)
    # time.sleep(30)
    # sparql = SPARQLWrapper(endpoint)
    # sparql.setQuery(prefixes + query)
    # sparql.setReturnFormat(JSON)
    # results = sparql.query().convert()
    # nbUp = int(results["results"]["bindings"][0]["nbID"]["value"])
    # results_df.at[i, "nb_UniProt"] = nbUp
    # results_df.at[i, "clique_size"] = (nbUp)*((nbUp-1)/2)
    # process.kill()
    # time.sleep(30)


    up_per_pathway = pd.read_csv(f"../../Results/UtilityFiles/{counter:02d}_UpPerPathway.csv", sep=",", header=0)
    list_proteins = set(up_per_pathway["entityID"].to_list())
    nb_prot = len(list_proteins)
    results_df.at[i, "nb_UniProt"] = nb_prot
    results_df.at[i, "clique_size"] = (nb_prot)*((nb_prot-1)/2)
    df = pd.read_csv(file, header=0, sep=",")
    provenances = df['provenanceScore'].to_list()
    nb_MICA = provenances.count("scoreMICA")
    nb_NSP = provenances.count("scoreNextStep")
    nb_hybrid = provenances.count("scoreHybrid")
    total_scores = nb_MICA+nb_NSP+nb_hybrid
    assert total_scores == len(df)
    results_df.at[i, "nb_scoresMICA"] = nb_MICA
    results_df.at[i, "nb_scoresNSP"] = nb_NSP
    results_df.at[i, "% NSP scores"] = (nb_NSP/total_scores)*100
    results_df.at[i, "nb_scoresHybrid"] = nb_hybrid

    counter += 1
    i += 1



Autophagy
CellCycle
CellCellCommunication
CellularResponseToStimuli
ChromatinOrganization
CircadianClock
DevelopmentalBiology
DigestionAndAbsorption
Disease
DNARepair
DNAReplication
DrugADME
ExtracellularMatrixOrganization
GeneExpression(Transcription)
Hemostasis
ImmuneSystem
Metabolism
MetabolismOfProteins
MetabolismOfRNA
MuscleContraction
NeuronalSystem
OrganelleBiogenesisAndMaintenance
ProgrammedCellDeath
ProteinLocalization
Reproduction
SensoryPerception
SignalTransduction
TransportOfSmallMolecules
VesicleMediatedTransport


In [6]:
results_df.to_csv("../../Results/PathwayComembership/OtherFiles/MetricsWeightedComembershipCliquesTopPathways.csv", sep=",", header=1, index=False)